<a href="https://colab.research.google.com/github/yaz455/aaase/blob/main/Day_2_Research_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -qU \
langgraph \
langchain \
langchain-openai \
langchain-tavily \
langchain-huggingface \
langchain-chroma \
chromadb \
sentence-transformers

In [2]:
import operator
from typing import Annotated, List, Dict
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

In [13]:
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")

llm = ChatOpenAI(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)

search_tool = TavilySearch(
    max_results=3,
    tavily_api_key=TAVILY_API_KEY
)

print("✅ LLM and Tavily initialized")


✅ LLM and Tavily initialized


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = Chroma(
    collection_name="enterprise_research_memory",
    embedding_function=embedding_model
)

print("✅ Vector Store initialized")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Vector Store initialized


In [11]:
import operator
from datetime import datetime
from typing import Annotated, List, Dict
from typing_extensions import TypedDict
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver


# ============================================================
# 1. AGENT STATE
# ============================================================

class AgentState(TypedDict):
    topic: str
    search_query: str
    collected_data: List[Dict]
    analyzed_data: List[Dict]
    quality_score: int
    iteration_count: int
    final_report: str
    execution_logs: Annotated[List[str], operator.add]


# ============================================================
# 2. QUALITY EVALUATOR
# ============================================================

class QualityScore(BaseModel):
    score: int = Field(ge=1, le=10)
    reasoning: str = Field(description="One-sentence justification")

import re

def evaluate_node(state: AgentState):

    prompt = f"""
Evaluate the overall quality of this research.

Topic:
{state["topic"]}

Research:
{state["analyzed_data"]}

Evaluate based on:
- Relevance
- Depth
- Source coverage
- Business usefulness
- Completeness

Return your answer exactly like this:

SCORE: 8
REASON: One short sentence explaining the score.

The score must be an integer from 1 to 10.
"""

    response = llm.invoke(prompt)

    text = response.content

    match = re.search(r"SCORE:\s*(10|[1-9])", text, re.IGNORECASE)

    if match:
        score = int(match.group(1))
    else:
        score = 5

    return {
        "quality_score": score,
        "execution_logs": [
            f"[{datetime.now()}] Quality Score: {score}/10. Raw evaluation: {text}"
        ]
    }

# ============================================================
# 3. COLLECT NODE
# ============================================================

def collect_node(state: AgentState):

    iteration = state["iteration_count"] + 1

    if iteration == 1:
        query = state["topic"]
    else:
        query = (
            f"{state['topic']} latest developments "
            f"enterprise applications risks best practices "
            f"research iteration {iteration}"
        )

    response = search_tool.invoke({
        "query": query
    })

    results = response.get("results", [])

    return {
        "search_query": query,
        "collected_data": results,
        "iteration_count": iteration,
        "execution_logs": [
            f"[{datetime.now()}] Iteration {iteration}: "
            f"collected {len(results)} sources."
        ]
    }


# ============================================================
# 4. MEMORY NODE
# ============================================================

def store_memory_node(state: AgentState):

    documents = []

    for item in state["collected_data"]:
        content = item.get("content", "")

        if content:
            documents.append(content)

    if documents:
        vector_store.add_texts(documents)

    return {
        "execution_logs": [
            f"[{datetime.now()}] "
            f"Stored {len(documents)} documents in vector memory."
        ]
    }


# ============================================================
# 5. ANALYSIS NODE
# ============================================================

def analyze_node(state: AgentState):

    analyzed_results = []

    for item in state["collected_data"]:

        content = item.get("content", "")

        if not content:
            continue

        prompt = f"""
You are an enterprise AI research analyst.

Research Topic:
{state["topic"]}

Source Content:
{content}

Analyze the source and provide:

1. Concise Summary
2. Key Findings
3. Business Impact
4. Potential Risks
5. Importance to the research topic

Keep the analysis professional and concise.
"""

        response = llm.invoke(prompt)

        analyzed_results.append({
            "source": item.get("url", "Unknown"),
            "analysis": response.content
        })

    return {
        "analyzed_data": analyzed_results,
        "execution_logs": [
            f"[{datetime.now()}] "
            f"Analyzed {len(analyzed_results)} sources."
        ]
    }




# ============================================================
# 7. REPORT NODE
# ============================================================

def report_node(state: AgentState):

    prompt = f"""
Generate a professional enterprise research report.

Topic:
{state["topic"]}

Research Analysis:
{state["analyzed_data"]}

Include:

1. Executive Summary
2. Key Findings
3. Business Impact
4. Risks
5. Opportunities
6. Strategic Recommendations
7. Conclusion
"""

    response = llm.invoke(prompt)

    return {
        "final_report": response.content,
        "execution_logs": [
            f"[{datetime.now()}] Final report generated."
        ]
    }


# ============================================================
# 8. AUDIT NODE
# ============================================================

def audit_node(state: AgentState):

    return {
        "execution_logs": [
            f"[{datetime.now()}] Workflow completed. "
            f"Iterations: {state['iteration_count']}. "
            f"Final Quality: {state['quality_score']}/10."
        ]
    }


# ============================================================
# 9. CONDITIONAL ROUTER
# ============================================================

def quality_router(state: AgentState) -> str:

    if state["quality_score"] >= 7:
        return "report"

    if state["iteration_count"] >= 3:
        return "report"

    return "collect"


# ============================================================
# 10. BUILD LANGGRAPH
# ============================================================

workflow = StateGraph(AgentState)

workflow.add_node("collect", collect_node)
workflow.add_node("store_memory", store_memory_node)
workflow.add_node("analyze", analyze_node)
workflow.add_node("evaluate", evaluate_node)
workflow.add_node("report", report_node)
workflow.add_node("audit", audit_node)

workflow.add_edge(START, "collect")

workflow.add_edge(
    "collect",
    "store_memory"
)

workflow.add_edge(
    "store_memory",
    "analyze"
)

workflow.add_edge(
    "analyze",
    "evaluate"
)

workflow.add_conditional_edges(
    "evaluate",
    quality_router,
    {
        "collect": "collect",
        "report": "report"
    }
)

workflow.add_edge(
    "report",
    "audit"
)

workflow.add_edge(
    "audit",
    END
)


# ============================================================
# 11. COMPILE AGENT
# ============================================================

app = workflow.compile(
    checkpointer=InMemorySaver()
)

print("✅ Enterprise Research AI Agent built successfully")

✅ Enterprise Research AI Agent built successfully


In [14]:
initial_state = {
    "topic": "Enterprise Agentic AI Systems",
    "search_query": "",
    "collected_data": [],
    "analyzed_data": [],
    "quality_score": 0,
    "iteration_count": 0,
    "final_report": "",
    "execution_logs": []
}

config = {
    "configurable": {
        "thread_id": "research-run-2"
    }
}

final_state = app.invoke(
    initial_state,
    config=config
)

print("✅ AGENT FINISHED")

✅ AGENT FINISHED


In [15]:
print(final_state["final_report"])

**ENTERPRISE RESEARCH REPORT**

**Topic:** Enterprise Agentic AI Systems: Architecture, Adoption Patterns, and Strategic Imperatives
**Classification:** Confidential – Strategic Planning
**Date:** October 26, 2023
**Prepared For:** Executive Leadership & Technology Strategy Office

---

### 1. Executive Summary

Enterprise Agentic AI represents a paradigm shift from **generative assistance** (copilots) to **autonomous execution** (digital labor). This research synthesizes market signals from platform vendors (Salesforce, ServiceNow), definitional frameworks, and Systems Integrator (SI) deployment realities (Capgemini) to define the current state of maturity.

**Core Thesis:** The enterprise market has entered **"Phase 1: Embedded Agency."** Agentic AI is achieving product-market fit not as a standalone horizontal platform, but as a native capability layer embedded within Systems of Record (CRM, ITSM). The immediate strategic imperative is not "build vs. buy" for agent frameworks, but *

In [16]:
for log in final_state["execution_logs"]:
    print(log)

[2026-07-20 09:46:21.500652] Iteration 1: collected 5 sources.
[2026-07-20 09:46:22.492645] Stored 5 documents in vector memory.
[2026-07-20 09:48:26.303502] Analyzed 5 sources.
[2026-07-20 09:48:54.998775] Quality Score: 8/10. Raw evaluation: SCORE: 8
REASON: The research offers high relevance, strong business-focused depth, and actionable insights across definition, architecture, and maturity models, but relies heavily on vendor and consultant secondary sources without independent analyst or academic validation.
[2026-07-20 09:52:45.879746] Iteration 1: collected 3 sources.
[2026-07-20 09:52:46.430070] Stored 3 documents in vector memory.
[2026-07-20 09:54:05.780183] Analyzed 3 sources.
[2026-07-20 09:55:27.612096] Quality Score: 7/10. Raw evaluation: SCORE: 7
REASON: The research offers strong relevance, depth, and business utility through complementary vendor, definitional, and SI perspectives, but is limited by narrow source coverage (only three vendor-aligned sources) and gaps in